# DMRG vs NQS — 3D toric code, L=3 OBC (N=54)

Collaborator DMRG reference (bosonic + fermionic, single-field axes) against the dual-basis NQS
(`nersc/launch_dmrg_L3_btc.sh`, weak-field wave: $|h|\le 0.2$ in tc3d convention). The reference is
already converted to tc3d conventions (`analysis/scripts/dmrg_reference.py`): $h_x^{\rm ours}=-h/2$,
$h_y^{\rm ours}=h_z^{\rm ours}=+h/2$, so the $x$ axis is plotted against $|h|$ and labelled $-h_x$.
Runs are joined by `analysis/scripts/dmrg_compare.py`; the notebook runs with no run data (DMRG lines only).
The acceptance rule `dmrg_compare.matches()` is deliberately left for the user to define.

## 1. Config

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "analysis/scripts"))
import dmrg_compare as dc

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

REF = ROOT / "results/dmrg_L3_obc/dmrg_L3_obc_reference.json"
RUN_DIRS = {"btc": "dmrg_L3_obc/btc", "ftc": "dmrg_L3_obc/ftc"}   # label -> path under ROOT/results
MODEL = ["bosonic", "fermionic"]
FIGS = ROOT / "analysis/figs"
XMAX = None   # e.g. 0.25 to zoom the plots on the weak-field wave

# (field key, x-axis label, sign so that x = sign * h_ours >= 0)
AXES = [("hx", r"$-h_x$", -1.0), ("hy", r"$h_y$", 1.0), ("hz", r"$h_z$", 1.0)]
OBS = [("sx_mean", "sx_nqs", r"$\langle\sigma^x\rangle$"), ("sy_mean", "sy_nqs", r"$\langle\sigma^y\rangle$"),
       ("sz_mean", "sz_nqs", r"$\langle\sigma^z\rangle$"), ("A_v_mean", "A_nqs", r"$\langle A_v\rangle$"),
       ("B_p_mean", "B_nqs", r"$\langle B_p\rangle$")]
ARM_COLOR = {a: plt.cm.plasma(v) for a, v in
             {"dual": 0.15, "primal": 0.65, "A": 0.15, "B": 0.5, "C": 0.8}.items()}
ARM_MARK = {"dual": "o", "primal": "s", "A": "o", "B": "s", "C": "^"}
def arm_style(arm):
    k = arm if arm in ARM_COLOR else arm[:1]
    return ARM_COLOR.get(k, "k"), ARM_MARK.get(k, "o")

## 2. Load reference and runs

In [ ]:
ref_rows = dc.load_ref(REF)
print(f"reference: {len(ref_rows)} rows from {REF.relative_to(ROOT)}")

run_dirs = []
for label, rel in RUN_DIRS.items():
    d = ROOT / "results" / rel
    if d.is_dir():
        run_dirs.append(d)
    else:
        print(f"  [{label}] {d.relative_to(ROOT)} missing -> no NQS points (DMRG only)")
TABLE = dc.load_table(run_dirs, REF) if run_dirs else []
print(f"NQS runs loaded: {len(TABLE)}  "
      + ", ".join(f"{m}={sum(r['model'] == m for r in TABLE)}" for m in MODEL))

def axis_of(o):
    # the single non-zero field of a reference/run point; h=0 belongs to every axis
    nz = [k for k in ("hx", "hy", "hz") if abs(o[k]) > 1e-12]
    return nz[0] if len(nz) == 1 else ("0" if not nz else None)

def dmrg_curve(model, axis, key):
    pts = [(r["ours"][axis], r["ours"][key]) for r in ref_rows
           if r["model"] == model and axis_of(r["ours"]) in (axis, "0")]
    pts.sort()
    return np.array([p[0] for p in pts]), np.array([p[1] for p in pts])

def nqs_points(model, axis, key, arm=None):
    rs = [r for r in TABLE if r["model"] == model and (arm is None or r["arm"] == arm)
          and axis_of(r) in (axis, "0") and r.get(key) is not None]
    rs.sort(key=lambda r: r[axis])
    return rs

def arms(model):
    return sorted({r["arm"] for r in TABLE if r["model"] == model})

## 3. Bosonic: energy vs DMRG

In [ ]:
def fig_energy(model, title):
    fig, axs = plt.subplots(3, 3, figsize=(11, 8.5))
    for i, (ax_key, xlab, sg) in enumerate(AXES):
        a_e, a_d, a_r = axs[i]
        h, E = dmrg_curve(model, ax_key, "E")
        a_e.plot(sg * h, E, "-", color="0.55", lw=1.2, zorder=1)
        a_e.plot(sg * h, E, ".", color="0.35", ms=5, zorder=2, label="DMRG")
        a_d.axhline(0, ls="--", color="0.3", lw=1)
        for arm in arms(model):
            c, mk = arm_style(arm)
            rs = nqs_points(model, ax_key, "E_nqs", arm)
            x = np.array([sg * r[ax_key] for r in rs])
            a_e.errorbar(x, [r["E_nqs"] for r in rs], [r["E_err"] or 0 for r in rs], fmt=mk, color=c,
                         ms=7, mfc="none", capsize=2, zorder=3, label=f"NQS {arm}")
            rd = [r for r in rs if r["dE"] is not None]
            a_d.errorbar([sg * r[ax_key] for r in rd], [r["dE"] for r in rd], [r["E_err"] or 0 for r in rd],
                         fmt=mk, color=c, ms=7, mfc="none", capsize=2)
            rr = [r for r in rd if r["dE"] != 0]
            a_r.plot([sg * r[ax_key] for r in rr], [abs(r["rel"]) for r in rr], mk, color=c, ms=7, mfc="none")
        a_r.set_yscale("log"); a_r.set_ylim(1e-9, 1e-1)
        if not arms(model):
            a_d.set_ylim(-1e-3, 1e-3)
        for a in (a_e, a_d, a_r):
            a.set_xlim(-0.01 * (XMAX or 1), XMAX)
        a_e.set_ylabel(r"$E$"); a_d.set_ylabel(r"$\Delta E=E_{\rm NQS}-E_{\rm DMRG}$")
        a_r.set_ylabel(r"$|\Delta E|/|E|$")
        for a in (a_e, a_d, a_r):
            a.set_xlabel(xlab)
        if i == 0:
            a_e.legend(frameon=False, fontsize=8)
    if not arms(model):
        axs[0, 1].set_title("no NQS runs loaded -- DMRG reference only", fontsize=9, color="0.4")
    fig.suptitle(title); fig.tight_layout()
    return fig

fig_energy("bosonic", "Bosonic 3D toric code, L=3 OBC")
# plt.savefig(FIGS / "dmrg_L3_bosonic_energy.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Observables vs DMRG

In [ ]:
# One row per axis; columns = the field-aligned <sigma>, the stabilizer(s) that field breaks,
# and |NQS - DMRG| / sigma_NQS. Observables the field leaves at 0 / 1 are not plotted (they are
# 0 / 1 to the DMRG digits in both methods and only show MCMC noise). Filled marker in the pull
# column = NQS above DMRG, open = below.
AXIS_OBS = {"hx": ("sx", [("B", r"$\langle B_p\rangle$")]),
            "hy": ("sy", [("A", r"$\langle A_v\rangle$"), ("B", r"$\langle B_p\rangle$")]),
            "hz": ("sz", [("A", r"$\langle A_v\rangle$")])}
OBS_KEY = {"sx": "sx_mean", "sy": "sy_mean", "sz": "sz_mean", "A": "A_v_mean", "B": "B_p_mean"}
SIG_LAB = {"sx": r"$\langle\sigma^x\rangle$", "sy": r"$\langle\sigma^y\rangle$", "sz": r"$\langle\sigma^z\rangle$"}
LANE_MARK = ["o", "s", "D", "^", "v"]

def _lanes(model):
    L = arms(model)
    return {a: (plt.cm.plasma(0.1 + 0.75 * i / max(len(L) - 1, 1)), LANE_MARK[i % len(LANE_MARK)])
            for i, a in enumerate(L)}

def _pts(model, ax_key, short, arm, sg):
    rs = nqs_points(model, ax_key, short + "_nqs", arm)
    x = np.array([sg * r[ax_key] for r in rs]); y = np.array([r[short + "_nqs"] for r in rs])
    e = np.array([r.get(short + "_err") or 0.0 for r in rs])
    ref = np.array([r.get(short + "_ref") if r.get(short + "_ref") is not None else np.nan for r in rs])
    return x, y, e, ref

def fig_obs(model, title, xmax=XMAX):
    lanes = _lanes(model)
    fig, axs = plt.subplots(3, 3, figsize=(13, 10))
    for i, (ax_key, xlab, sg) in enumerate(AXES):
        sig, stabs = AXIS_OBS[ax_key]
        a_s, a_st, a_p = axs[i]
        panels = [(a_s, [(sig, SIG_LAB[sig])]), (a_st, stabs)]
        for a, obs in panels:
            for k, (short, lab) in enumerate(obs):
                h, v = dmrg_curve(model, ax_key, OBS_KEY[short])
                keep = h <= (xmax or np.inf) * 1.05
                a.plot(sg * h[keep], v[keep], ("o-", "s--")[k], color=("0.45", "0.65")[k], ms=4, lw=1.2,
                       label="DMRG" + (f" {lab}" if len(obs) > 1 else ""), zorder=1)
                for arm, (c, mk) in lanes.items():
                    x, y, e, ref = _pts(model, ax_key, short, arm, sg)
                    if not len(x):
                        continue
                    a.errorbar(x, y, e, fmt=mk, color=c, ms=7, mfc="none", mew=1.3, capsize=3,
                               lw=1, zorder=3, label=f"NQS {arm}" if k == 0 else None)
                    ok = (e > 0) & np.isfinite(ref)
                    pull = (y[ok] - ref[ok]) / e[ok]
                    up = pull > 0
                    dx = (0.0 if a is a_s else 0.004 * (k + 1)) * (xmax or 1) / 0.25   # offset per observable
                    for sel, fc in ((up, c), (~up, "none")):
                        a_p.plot(x[ok][sel] + dx, np.abs(pull[sel]), mk, color=c, mfc=fc, ms=7, mew=1.3, zorder=3)
            a.set_ylabel(" / ".join(lab for _, lab in obs) if len(obs) > 1 else obs[0][1])
            a.set_xlim(-0.01 * (xmax or 1), xmax)
        a_p.set_yscale("log"); a_p.set_ylim(0.03, 3e3)
        a_p.axhline(1, color="0.6", lw=0.8, ls="--"); a_p.axhline(3, color="0.6", lw=0.8, ls=":")
        a_p.text(0.98, 0.92, r"filled: NQS $>$ DMRG; stabilizer points offset right",
                 transform=a_p.transAxes, ha="right", fontsize=8, color="0.3")
        a_p.set_ylabel(r"$|{\rm NQS}-{\rm DMRG}|/\sigma_{\rm NQS}$"); a_p.set_xlim(-0.01 * (xmax or 1), xmax)
        for a in (a_s, a_st, a_p):
            a.set_xlabel(xlab)
        if len(stabs) > 1:
            a_st.legend([h for h in a_st.get_legend_handles_labels()[0] if "DMRG" in str(h.get_label())],
                        [l for l in a_st.get_legend_handles_labels()[1] if "DMRG" in l],
                        frameon=False, fontsize=8, loc="lower left")
    # one figure-wide legend: every lane that has points on ANY axis (a lane run on one axis
    # only, e.g. B on hz, would otherwise never appear)
    from matplotlib.lines import Line2D
    handles = [Line2D([], [], marker="o", ls="-", color="0.45", ms=4, label="DMRG")]
    handles += [Line2D([], [], marker=mk, ls="", color=c, mfc="none", mew=1.3, ms=7, label=f"NQS {arm}")
                for arm, (c, mk) in lanes.items()]
    fig.legend(handles=handles, loc="upper center", ncol=min(len(handles), 6), frameon=False, fontsize=9,
               bbox_to_anchor=(0.5, 0.97))
    fig.suptitle(title, y=0.995); fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig

fig_obs("bosonic", "Bosonic observables that respond to the field, L=3 OBC", xmax=0.25)
# plt.savefig(FIGS / "dmrg_L3_bosonic_observables.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Fermionic: arms A / B / C

In [ ]:
fig_energy("fermionic", "Fermionic 3D toric code, L=3 OBC  (arms: A sign-frame, B analytic head, C blind)")
# plt.savefig(FIGS / "dmrg_L3_fermionic_energy.png", dpi=300, bbox_inches="tight")
plt.show()

fig_obs("fermionic", "Fermionic observables that respond to the field, L=3 OBC", xmax=0.25)
# plt.savefig(FIGS / "dmrg_L3_fermionic_observables.png", dpi=300, bbox_inches="tight")
plt.show()

try:
    import pandas as pd
    ferm = [r for r in TABLE if r["model"] == "fermionic" and r["rel"] is not None]
    if ferm:
        df = pd.DataFrame(ferm)
        df["point"] = [f"hx={r.hx:+.2f} hy={r.hy:.2f} hz={r.hz:.2f}" for r in df.itertuples()]
        print("relative error |dE|/|E| per arm per point")
        print(df.assign(rel=df.rel.abs()).pivot_table(index="point", columns="arm", values="rel").to_string())
    else:
        print("no fermionic runs with a reference match yet")
except ImportError:
    print("pandas not installed; see the verdict table below")

## 6. Verdict table

In [ ]:
if TABLE:
    print(dc.format_table(TABLE))
    n_y = sum(r["match"] is True for r in TABLE); n_n = sum(r["match"] is False for r in TABLE)
    print(f"\nmatch: {n_y} yes / {n_n} no / {sum(r['match'] is None for r in TABLE)} undecided "
          "(define dmrg_compare.matches() to enable)")
else:
    print("no NQS runs loaded yet. Reference points of the weak-field wave (ours, bosonic):")
    for r in ref_rows:
        o = r["ours"]
        if r["model"] == "bosonic" and max(abs(o[k]) for k in ("hx", "hy", "hz")) <= 0.2 + 1e-9 \
                and axis_of(o) not in ("0", None):
            print(f"  hx={o['hx']:+.2f} hy={o['hy']:.2f} hz={o['hz']:.2f}   E_DMRG={o['E']:.6f}")